In [60]:
import os
import requests

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-04.parquet"
file_path = "../data/yellow_tripdata_2026-04.parquet"

# Baixa o arquivo apenas se ele não existir
if not os.path.exists(file_path):
    print("Baixando dataset...")
    response = requests.get(url)
    with open(file_path, "wb") as f:
        f.write(response.content)
    print("Download concluído!")
else:
    print("Arquivo já existe na pasta data/.")

Arquivo já existe na pasta data/.


In [61]:
import pandas as pd
import duckdb
import time

file_path = "../data/raw/yellow_tripdata_2026-01.parquet"

print("Iniciando o teste de performance...\n")

# ==========================================
# 1. ABORDAGEM COM PANDAS
# ==========================================
start_pandas = time.time()

# O Pandas precisa carregar o arquivo INTEIRO para a memória RAM primeiro
df_pandas = pd.read_parquet(file_path)

# Só depois ele faz o filtro e a agregação
df_pandas['data_corrida'] = df_pandas['tpep_pickup_datetime'].dt.date
resultado_pandas = (
    df_pandas[
        (df_pandas['tpep_pickup_datetime'] >= '2026-04-01') & 
        (df_pandas['tpep_pickup_datetime'] < '2026-05-01')
    ]
    .groupby('data_corrida')
    .agg(
        total_corridas=('tpep_pickup_datetime', 'count'),
        ticket_medio=('fare_amount', 'mean')
    )
)

tempo_pandas = time.time() - start_pandas
print(f"Tempo Pandas: {tempo_pandas:.4f} segundos")


# ==========================================
# 2. ABORDAGEM COM DUCKDB
# ==========================================
start_duckdb = time.time()

# O DuckDB varre o arquivo no disco, não carrega tudo na RAM,
# e executa a agregação diretamente em C++ otimizado usando SQL.
query = f"""
    SELECT 
        CAST(tpep_pickup_datetime AS DATE) AS data_corrida,
        COUNT(*) AS total_corridas,
        AVG(fare_amount) AS ticket_medio
    FROM '{file_path}'
    WHERE tpep_pickup_datetime >= '2026-04-01' 
      AND tpep_pickup_datetime < '2026-05-01'
    GROUP BY 1
"""
resultado_duckdb = duckdb.query(query).df()

tempo_duckdb = time.time() - start_duckdb
print(f"Tempo DuckDB: {tempo_duckdb:.4f} segundos")

# ==========================================
print("\n--- RESULTADO ---")
if tempo_duckdb < tempo_pandas:
    print(f"DuckDB foi {tempo_pandas / tempo_duckdb:.1f}x mais rápido!")
else:
    print("Pandas foi mais rápido neste dataset pequeno.")

Iniciando o teste de performance...

Tempo Pandas: 1.1366 segundos
Tempo DuckDB: 0.0404 segundos

--- RESULTADO ---
DuckDB foi 28.1x mais rápido!


In [62]:
resultado_pandas.head()

,total_corridas,ticket_medio
data_corrida,,
2026-04-01,118268,21.064097
2026-04-02,123121,20.086183
2026-04-03,110948,19.993483
2026-04-04,123921,19.645618
2026-04-05,106595,20.278599


In [63]:
# Cria uma conexão persistente local
con = duckdb.connect('nyc_taxi.duckdb')

In [64]:
# Checando o schema
con.execute(f"DESCRIBE SELECT * FROM '{file_path}'").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [65]:
con.execute(f"SELECT * FROM '{file_path}' limit 15").df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2026-04-01 00:40:05,2026-04-01 00:52:44,1,2.80,1,N,237,68,1,15.6,4.25,0.5,4.25,0.00,1.0,25.60,2.5,0.0,0.75
1,2,2026-04-01 00:09:19,2026-04-01 00:21:29,1,7.37,1,N,138,75,1,28.2,6.00,0.5,9.03,7.46,1.0,54.19,0.0,2.0,0.00
2,2,2026-04-01 00:15:29,2026-04-01 00:34:14,1,7.66,1,N,138,112,1,31.7,6.00,0.5,5.00,0.00,1.0,46.20,0.0,2.0,0.00
3,1,2026-04-01 00:14:20,2026-04-01 00:27:49,0,7.90,1,N,138,262,1,31.0,10.50,0.5,10.09,7.46,1.0,60.55,2.5,2.0,0.00
4,2,2026-04-01 00:04:53,2026-04-01 00:11:54,1,1.34,1,N,230,234,1,8.6,1.00,0.5,2.87,0.00,1.0,17.22,2.5,0.0,0.75
5,7,2026-04-01 00:10:12,2026-04-01 00:10:12,2,1.38,1,N,140,262,2,9.3,0.00,0.5,0.00,0.00,1.0,14.30,2.5,0.0,0.00
6,2,2026-04-01 00:37:00,2026-04-01 00:45:17,1,1.74,1,N,68,107,1,10.7,1.00,0.5,4.93,0.00,1.0,21.38,2.5,0.0,0.75
7,2,2026-04-01 00:01:45,2026-04-01 00:11:27,1,2.68,1,N,236,41,1,13.5,1.00,0.5,1.85,0.00,1.0,20.35,2.5,0.0,0.00
8,2,2026-04-01 00:30:02,2026-04-01 00:38:59,3,1.38,1,N,161,246,2,10.0,1.00,0.5,0.00,0.00,1.0,15.75,2.5,0.0,0.75
9,1,2026-04-01 00:14:55,2026-04-01 00:24:35,1,2.40,1,N,142,68,1,12.8,4.25,0.5,3.70,0.00,1.0,22.25,2.5,0.0,0.75


In [66]:
# Limpeza e agregação: Filtrando viagens inválidas e agregando por dia
query_limpeza = f"""
    SELECT 
        tpep_pickup_datetime::DATE AS data_viagem,
        COUNT(*) AS total_viagens,
        AVG(total_amount) AS ticket_medio,
        AVG(trip_distance) AS distancia_media
    FROM '{file_path}'
    WHERE total_amount > 0 
      AND trip_distance > 0
      AND tpep_pickup_datetime >= '2026-04-01' 
      AND tpep_pickup_datetime < '2026-05-01'
    GROUP BY 1
    ORDER BY 1
"""

# Executa a query e salva em um DataFrame para visualizar
df_agregado = con.execute(query_limpeza).df()

In [67]:
df_agregado.head()

,data_viagem,total_viagens,ticket_medio,distancia_media
0,2026-04-01,114087,30.116389,5.320934
1,2026-04-02,119244,29.199633,3.890518
2,2026-04-03,106930,28.805334,4.755335
3,2026-04-04,122115,27.579536,3.783667
4,2026-04-05,104887,28.438238,8.739206


In [68]:
query_exportacao = f"""
    COPY ({query_limpeza}) 
    TO '../data/processed/viagens_diarias_2026_04.parquet' 
    (FORMAT PARQUET);
"""
con.execute(query_exportacao)
con.close()

In [70]:
import pandas as pd
import plotly.express as px

# Lê o dado processado
df_final = pd.read_parquet('../data/processed/viagens_diarias_2026_04.parquet')

# Plota a tendência
fig = px.line(df_final, x='data_viagem', y='total_viagens', 
              title='Volume de Viagens de Táxi por Dia - NYC (Abril 2026)')
fig.show()
